#Import

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

#Read Bronze table

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")

#Silver Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
     

##Normalize Maintenance Flag to Boolean

In [0]:

# Convert maintenance values from YES/NO to boolean.
df = df.withColumn(
    "maintenance",
    when(upper(col("maintenance")) == "YES", lit(True))
     .when(upper(col("maintenance")) == "NO", lit(False))
     .otherwise(None)
)


##Renaming Columns

In [0]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity checks of dataframe

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_product_category")

##Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_product_category LIMIT 10